# مترجم خودکار مانگا / مانهوا به فارسی

OCR → پاک‌سازی حباب → ترجمه با Gemini → رندر فارسی

**ترتیب:** سلول‌ها را از بالا به پایین با Shift+Enter اجرا کنید.

قبل از شروع: `Runtime → Change runtime type → GPU (T4)`

## ۴) نوشتن اسکریپت اصلی مترجم
این سلول فایل `manga_translator.py` رو می‌سازه (همون پایپ‌لاین: تشخیص متن -> پاکسازی -> ترجمه -> بازنویسی).

In [17]:
!git clone https://github.com/amirwolf5122/Manga-AutoTranslate.git
!cp Manga-AutoTranslate/manga_translator.py .

Cloning into 'Manga-AutoTranslate'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 42 (delta 12), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 2.94 MiB | 6.41 MiB/s, done.
Resolving deltas: 100% (12/12), done.


## ۱) نصب پیش‌نیازها

In [13]:

!pip install numpy==1.26.4

!pip install --no-deps PyMuPDF
!pip install --no-deps opencv-python-headless Pillow google-genai arabic-reshaper python-bidi requests beautifulsoup4

!pip install --no-deps imgaug
!pip install --no-deps scipy imageio matplotlib networkx shapely
# یا برای CPU
#!pip install --no-deps paddlepaddle-gpu==2.6.1 lmdb
!pip install --no-deps paddlepaddle==2.6.2 lmdb astor pyclipper
!pip install --no-deps paddleocr==2.7.0.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 14.3 MB/s eta 0:00:00


## ۲) دانلود فونت فارسی (Vazirmatn)
برای اینکه متن فارسی درست نمایش داده بشه، یک فونت پشتیبان فارسی لازم داریم.

In [3]:
import os
os.makedirs('fonts', exist_ok=True)
!wget -q -O fonts/Vazirmatn-Bold.ttf \
  https://github.com/rastikerdar/vazirmatn/raw/master/fonts/ttf/Vazirmatn-Bold.ttf
print('فونت دانلود شد:', os.path.isfile('fonts/Vazirmatn-Bold.ttf'))

فونت دانلود شد: True


## ۳) کلید Gemini API
کلید رایگان‌تون رو از [aistudio.google.com/api-keys](https://aistudio.google.com/api-keys) بگیرید و اینجا (به‌صورت مخفی) وارد کنید.

In [7]:
from getpass import getpass
import os

print("کلیدهای Gemini رو یکی‌یکی وارد کن (خالی بذار تا تموم بشه):")
keys = []
while True:
    k = getpass(f"کلید {len(keys)+1} (Enter = پایان): ").strip()
    if not k:
        break
    keys.append(k)

if not keys:
    raise SystemExit("حداقل یک کلید لازم است.")

os.environ["GEMINI_API_KEY"] = ",".join(keys)
print(f"{len(keys)} کلید ثبت شد.")

کلیدهای Gemini رو یکی‌یکی وارد کن (خالی بذار تا تموم بشه):
کلید 1 (Enter = پایان): ··········
کلید 2 (Enter = پایان): ··········
کلید 3 (Enter = پایان): ··········
کلید 4 (Enter = پایان): ··········
کلید 5 (Enter = پایان): ··········
کلید 6 (Enter = پایان): ··········
کلید 7 (Enter = پایان): ··········
6 کلید ثبت شد.


زبان اصلی متن منبع رو انتخاب کنید (این مهمه؛ انتخاب اشتباه باعث می‌شه OCR متن رو درست استخراج نکنه):

## ۵) ورودی رو بدید و ترجمه رو اجرا کنید
این سلول خودش تشخیص می‌ده که چی بهش دادید:
- اگه یک **لینک** (http/https) وارد کنید، تصاویر همون صفحه خودکار دانلود می‌شن.
- اگه Enter بزنید، پنجره‌ی آپلود باز می‌شه؛ می‌تونید یک فایل **.zip**، یک فایل **.pdf**، یا چند تا **تصویر** (jpg/png/...) رو هم‌زمان انتخاب کنید — نوعش خودکار تشخیص داده می‌شه.

In [13]:
import os
from google.colab import files

INPUT_DIR = 'input_pages'
os.makedirs(INPUT_DIR, exist_ok=True)

url = input('اگه لینک صفحه دارید وارد کنید (وگرنه Enter بزنید تا فایل آپلود کنید): ').strip()

if url.lower().startswith('http://') or url.lower().startswith('https://'):
    input_path = url
    print(f'از لینک استفاده می‌شه: {input_path}')
else:
    print('فایل(ها) رو انتخاب کنید (یک zip، یک pdf، یا چند تصویر):')
    uploaded = files.upload()
    names = list(uploaded.keys())

    if len(names) == 1 and names[0].lower().endswith('.zip'):
        input_path = names[0]
        with open(input_path, 'wb') as f:
            f.write(uploaded[names[0]])
        print(f'فایل zip شناسایی و ذخیره شد: {input_path}')

    elif len(names) == 1 and names[0].lower().endswith('.pdf'):
        input_path = names[0]
        with open(input_path, 'wb') as f:
            f.write(uploaded[names[0]])
        print(f'فایل pdf شناسایی و ذخیره شد: {input_path}')

    else:
        for name, data in uploaded.items():
            with open(os.path.join(INPUT_DIR, name), 'wb') as f:
                f.write(data)
        input_path = INPUT_DIR
        print(f'{len(names)} تصویر آپلود و در پوشه‌ی {INPUT_DIR} ذخیره شد.')

print('\nورودی نهایی برای پردازش:', input_path)

اگه لینک صفحه دارید وارد کنید (وگرنه Enter بزنید تا فایل آپلود کنید): https://asurascans.com/comics/my-slain-dragon-bride-00dcbf97/chapter/9
از لینک استفاده می‌شه: https://asurascans.com/comics/my-slain-dragon-bride-00dcbf97/chapter/9

ورودی نهایی برای پردازش: https://asurascans.com/comics/my-slain-dragon-bride-00dcbf97/chapter/9


اگه متن اصلی مانگا ژاپنیه، `--ocr-lang ja en` رو نگه دارید. برای کره‌ای `ko`، برای انگلیسی فقط `en`.
برای کمیک غربی/کره‌ای که چپ‌به‌راست خونده می‌شه، `--reading-order` رو به `ltr` تغییر بدید.
خروجی پیش‌فرض یک فایل zip می‌سازه؛ اگه یک PDF یکجا می‌خواید، `output_path` رو به `'output_pages_fa.pdf'` تغییر بدید.

In [14]:
#--api-key KEY1 --api-key KEY2 --api-key KEY3
# یا
#--api-key "KEY1,KEY2,KEY3"
# یا
#export GEMINI_API_KEY="KEY1,KEY2,KEY3"
output_path = 'output_pages_fa.pdf'  # یا 'output_pages_fa.zip'

!python manga_translator.py \
  -i "{input_path}" \
  -o "{output_path}" \
  --font fonts/Vazirmatn-Bold.ttf \
  --ocr-lang en \
  --reading-order rtl

# نکته: --ocr-lang باید با زبانی که روی خود تصویر چاپ شده یکی باشه.
# اگه فایلتون از قبل انگلیسی اسکنلیشن شده (اکثر ریلیزهای معروف اینطورن)،
# 'en' درسته. برای اسکن خام ژاپنی از 'ja en' و برای کره‌ای از 'ko en' استفاده کنید.

# اگه این سلول قطع شد (مثلاً سهمیه‌ی روزانه‌ی Gemini تموم شد)، فقط دوباره
# همین سلول رو اجرا کنید؛ صفحاتی که قبلاً ترجمه شدن دوباره پردازش نمی‌شن.

[*] GPU شناسایی نشد؛ OCR روی CPU اجرا می‌شه و کندتره. اگه توی Colab هستی و GPU داری، از منوی Runtime > Change runtime type یه GPU (مثلاً T4) انتخاب کن.
[*] در حال بارگذاری مدل PaddleOCR برای زبان(های) ['en'] (gpu=False) ...
[*] مدل PaddleOCR با زبان 'en' و دستگاه 'cpu' بارگذاری شد.
[*] مدل ترجمه: gemini-flash-latest | 6 کلید API (جابه‌جایی خودکار هنگام اتمام سهمیه)
[*] دانلود تصاویر از لینک: https://asurascans.com/comics/my-slain-dragon-bride-00dcbf97/chapter/9
    24 تصویر از https://asurascans.com/comics/my-slain-dragon-bride-00dcbf97/chapter/9 دانلود شد.
-------------------- شروع عملیات جدید --------------------
[فاز ۱ - OCR] شروع استخراج متن...
- پردازش 'page_001.webp'...
    [>] OCR موازی تیکه‌ی 1 (ردیف 0 تا 600)
[2026/08/06 09:00:58] ppocr WARNING: Since the angle classifier is not initialized, it will not be used during the forward process
[2026/08/06 09:00:59] ppocr WARNING: Since the angle classifier is not initialized, it will not be used during the forward process
[2026/08/0

## ۷) دانلود خروجی‌ها

In [12]:
from google.colab import files
import os
import zipfile
from IPython.display import display
from PIL import Image as PILImage

print('انتخاب کنید:')
print('1) دانلود ترجمه')
print('2) نمایش ترجمه')
choice = input('عدد رو وارد کنید [پیش‌فرض 1]: ').strip() or '1'

if str(choice) == "1":
    files.download(output_path)
else:
    if not os.path.exists(output_path):
        print(f"❌ فایل یافت نشد: {output_path}")

    elif output_path.endswith('.pdf'):
        from IPython.display import IFrame
        print("📄 در حال نمایش فایل PDF...")
        display(IFrame(output_path, width=800, height=600))

    elif output_path.endswith('.zip'):
        print("📦 در حال استخراج و نمایش عکس‌های درون ZIP...")
        extract_folder = 'temp_extracted_imgs'
        os.makedirs(extract_folder, exist_ok=True)

        with zipfile.ZipFile(output_path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

        valid_extensions = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
        image_files = []

        for root, _, files_list in os.walk(extract_folder):
            for f in files_list:
                if f.lower().endswith(valid_extensions):
                    image_files.append(os.path.join(root, f))

        image_files.sort()

        if image_files:
            print(f"✅ تعداد {len(image_files)} تصویر پیدا شد:\n")
            for img_path in image_files:
                print(f"📌 {os.path.basename(img_path)}")
                img = PILImage.open(img_path)
                display(img)
                print("-" * 40)
        else:
            print("⚠️ هیچ تصویری داخل فایل زیپ پیدا نشد.")

    else:
        print("❓ فرمت فایل پشتیبانی نمی‌شود (فقط .pdf یا .zip).")

انتخاب کنید:
1) دانلود ترجمه
2) نمایش ترجمه
عدد رو وارد کنید [پیش‌فرض 1]: 1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## نکات
- اولین اجرا کمی طول می‌کشه چون `easyocr` مدل‌هاش رو دانلود می‌کنه.
- اگه از منوی Colab یک GPU فعال کنید (Runtime > Change runtime type > GPU)، می‌تونید در فراخوانی `MangaTranslator` پارامتر `gpu=True` رو در اسکریپت (یا با اضافه کردن `--gpu` به دستور بالا) فعال کنید تا OCR سریع‌تر بشه.
- اگه دقت OCR روی فونت‌های افکتی ژاپنی راضی‌کننده نبود، می‌شه مرحله‌ی OCR رو با `manga-ocr` جایگزین کرد.
- برای پس‌زمینه‌های شلوغ، پاکسازی `cv2.inpaint` ممکنه کامل نباشه؛ در این صورت به یک مدل inpainting قوی‌تر (مثل LaMa) نیاز دارید.